# 03 · Food Agent-to-Agent Handoff
Food Researcher가 근거 브리프를 작성해 Launch Advisor에게 명시적으로 전달하고, Advisor가 출시 검토 권고를 만듭니다.

**학습 목표:** 공개 코어 API `run_food_handoff()`와 sender·recipient·type·payload가 있는 handoff artifact를 확인합니다.  
**실행 경계:** 두 Agent는 AI Core LLM만 사용하며 MCP Tool·외부 쓰기 작업은 수행하지 않습니다.  
**환경 변수:** 실제 AI Core LLM을 사용하므로 루트 `.env` 설정이 필요합니다.

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #2e7d32; background: #e8f5e9; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #2e7d32; letter-spacing: .02em;">🖥️  UI 실행</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">UI: Food Agent-to-Agent Handoff</h2>
<p style="margin: 7px 0;">Researcher와 Advisor의 실제 LLM handoff artifact 및 trace를 대화와 JSON으로 확인합니다. 외부 시스템은 변경하지 않습니다.</p>
  </div>
</div>


In [ ]:
from pathlib import Path
import sys
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'shared').exists())
sys.path.insert(0, str(project_root / 'shared'))
sys.path.insert(0, str(project_root / '4_agent_patterns'))

from src.a2a_ui import launch_a2a_ui
a2a_ui = launch_a2a_ui()


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<ul style="margin: 7px 0 0; padding-left: 1.3em;"><li>UI에서 Researcher → Advisor handoff artifact를 확인한다.</li><li>두 Agent의 System Prompt와 Tool 경계를 확인한다.</li><li>아래 코드는 같은 공개 API를 살펴보는 선택 심화 자료다.</li></ul>
  </div>
</div>


<hr style="border: 0; border-top: 2px solid #c7d2e1; margin: 32px 0 14px;">
<div style="margin: 0 0 18px; color: #4b5f78; font-size: .86em; font-weight: 700; letter-spacing: .04em;">🧩 아래부터 코드 베이스 실습</div>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">선택: 코드로 같은 작업 보기</h2><p style="margin: 7px 0;">노트북은 구현을 복사하지 않고 공개 코어 API만 호출합니다.</p></div></div>


In [ ]:
from pathlib import Path
import sys
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'shared').exists())
sys.path.insert(0, str(project_root / 'shared'))
sys.path.insert(0, str(project_root / '4_agent_patterns'))
from src.a2a import list_a2a_agents, run_food_handoff


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. A2A Agent 설정 보기</h2><p style="margin: 7px 0;">공개 카탈로그로 각 Agent의 System Prompt, Tool, 실행 경계를 확인합니다.</p></div></div>


In [ ]:
for agent in list_a2a_agents():
    print(f"## {agent['agent_label']} ({agent['agent_id']})")
    print(agent['system_prompt'])
    print(f"Tools: {agent['tools'] or '없음'}\n실행 경계: {agent['execution_boundary']}\n")


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">2. Handoff 실행과 trace</h2><p style="margin: 7px 0;">공개 API가 반환한 handoff artifact와 최종 recommendation을 확인합니다.</p></div></div>


In [ ]:
result = run_food_handoff('고단백 닭가슴살 도시락 출시를 검토해줘.')
print(result['report']['payload']['recommendation'])
for event in result['trace']:
    print(event)
